[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/23_cross_attention.ipynb)

# 🟠 Medium: Multi-Head Cross-Attention

Implement **multi-head cross-attention** (encoder-decoder attention).

### Signature
```python
class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model: int, num_heads: int): ...
    def forward(self, x_q: Tensor, x_kv: Tensor) -> Tensor:
        # x_q: (B, S_q, D) — decoder queries
        # x_kv: (B, S_kv, D) — encoder keys/values
```

### Key Differences from Self-Attention
- Q comes from the decoder, K and V come from the encoder
- No causal mask (all encoder positions visible)

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [ ]:
import torch
import torch.nn as nn
import math

In [ ]:
# ✏️ YOUR IMPLEMENTATION HERE

class MultiHeadCrossAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        self.d_model = d_model
        self.num_heads = num_heads

    def forward(self, x_q, x_kv):
      B, S_q, D = x_q.shape[0], x_q.shape[1], x_q.shape[2]
      S_kv = x_kv.shape[1]

      d_head = self.d_model // self.num_heads
      Q = self.W_q(x_q).reshape(B, S_q, self.num_heads, d_head).transpose(1,2)
      K = self.W_k(x_kv).reshape(B, S_kv, self.num_heads, d_head).transpose(1,2)
      V = self.W_v(x_kv).reshape(B, S_kv, self.num_heads, d_head).transpose(1,2)

      score = Q @ K.transpose(-1,-2) / math.sqrt(d_head)

      score = torch.softmax(score, dim=-1)

      out =  score @ V
      out = out.transpose(1,2).reshape(B, S_q, D)

      return self.W_o(out)






        pass  # Q from x_q, K/V from x_kv, no causal mask

In [ ]:
# 🧪 Debug
attn = MultiHeadCrossAttention(64, 4)
x_q = torch.randn(2, 6, 64)
x_kv = torch.randn(2, 10, 64)
print('Output:', attn(x_q, x_kv).shape)

In [ ]:
# ✅ SUBMIT
from torch_judge import check
check('cross_attention')